# ArcNeuron on Google Colab

This notebook only runs the real repository files. `train.py` measures the actual data and resolves context, batch size, and training steps; the notebook no longer hardcodes a 3000-step run.

Enable a GPU from **Runtime → Change runtime type → GPU** before running.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = "https://github.com/ArcatureLabs/ArcNeuron.git"
ROOT = Path("/content/ArcNeuron")

if not (ROOT / "arcneuron.py").is_file():
    if ROOT.exists():
        subprocess.run(["rm", "-rf", str(ROOT)], check=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(ROOT)], check=True)

os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"], check=True)

def run_live(command):
    """Run a child Python process and stream every output line into this Colab cell."""
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, command)

print("working directory:", Path.cwd())


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select Runtime > Change runtime type > GPU and run again.")

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", torch.cuda.is_bf16_supported())
props = torch.cuda.get_device_properties(0)
print(f"VRAM: {props.total_memory / 1024**3:.1f} GiB")


## Model configuration

Only architecture decisions stay here. Training budget belongs to `train.py` and is calculated from `train.txt`.


In [ ]:
VOCAB_SIZE = 1024
DIM = 384
HEADS = 6
KV_HEADS = 2
FFN_DIM = 1024
PRELUDE_LAYERS = 1
CORE_LAYERS = 2
CODA_LAYERS = 1
MAX_DEPTH = 4

BASE_CKPT = "arcneuron.pt"
TUNED_CKPT = "arcneuron-tuned.pt"


## Train base model

The trainer prints parameter count, token count, context, batch size, steps, corpus-equivalent exposure, loss, validation loss, and ETA live.


In [ ]:
train_command = [
    sys.executable, "-u", "train.py",
    "--data", "train.txt",
    "--out", BASE_CKPT,
    "--steps", "auto",
    "--batch-size", "auto",
    "--context", "auto",
    "--vocab-size", str(VOCAB_SIZE),
    "--dim", str(DIM),
    "--heads", str(HEADS),
    "--kv-heads", str(KV_HEADS),
    "--ffn-dim", str(FFN_DIM),
    "--prelude-layers", str(PRELUDE_LAYERS),
    "--core-layers", str(CORE_LAYERS),
    "--coda-layers", str(CODA_LAYERS),
    "--max-depth", str(MAX_DEPTH),
]

run_live(train_command)


## Tuning

`tune.py` also derives its step budget from the real `tune.txt`; there is no fixed 600/1000-step constant anymore.


In [ ]:
tune_command = [
    sys.executable, "-u", "tune.py",
    "--checkpoint", BASE_CKPT,
    "--data", "tune.txt",
    "--replay-data", "train.txt",
    "--out", TUNED_CKPT,
    "--steps", "auto",
    "--batch-size", "auto",
    "--context", "auto",
    "--max-depth", str(MAX_DEPTH),
]

run_live(tune_command)


## Generate


In [ ]:
from generate import load_model, generate

device = torch.device("cuda")
checkpoint = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
model, tokenizer = load_model(checkpoint, device)

PROMPT = "If a cat loses one leg, is it still a mammal? Explain."
DEPTH = 4

answer = generate(
    model=model,
    tokenizer=tokenizer,
    prompt=PROMPT,
    depth=DEPTH,
    max_new_tokens=128,
    temperature=0.70,
    top_k=40,
    top_p=0.90,
    repetition_penalty=1.08,
    repeat_window=96,
    include_prompt=False,
    device=device,
)

print("Prompt:", PROMPT)
print("Answer:", answer)


## Compare recurrent depth


In [ ]:
for depth in [1, 2, 4, 8]:
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    answer = generate(
        model=model,
        tokenizer=tokenizer,
        prompt=PROMPT,
        depth=depth,
        max_new_tokens=128,
        temperature=0.0,
        top_k=0,
        top_p=1.0,
        repetition_penalty=1.06,
        repeat_window=96,
        include_prompt=False,
        device=device,
    )
    print(f"\n{'=' * 24} depth={depth} {'=' * 24}\n")
    print(answer)


## Download checkpoint


In [ ]:
from google.colab import files
path = TUNED_CKPT if Path(TUNED_CKPT).is_file() else BASE_CKPT
files.download(path)
